# 🎭 Konuşma Duygu Tanıma (Speech Emotion Recognition)
Bu notebook, RAVDESS veri seti üzerinde **CNN–LSTM** mimarisi kullanarak konuşma duygu tanıma (SER) görevini gerçekleştirir.

## Adımlar
1. Kurulum
2. Veri İndirme ve Ön İşleme
3. Özellik Çıkarımı (Mel-spectrogram + Delta)
4. Dataset & DataLoader Tanımı
5. Model (CNN–LSTM) Tanımı
6. Eğitim Döngüsü
7. Değerlendirme


In [10]:
# 1. Kurulum
#!pip install -q torch torchvision torchaudio librosa numpy pandas sklearn soundfile

In [2]:
# 2. Veri İndirme ve Ön İşleme
import os, glob
import pandas as pd
import librosa
#!wget -q --show-progress https://zenodo.org/record/1188976/files/Audio_Speech_Actors_01-24.zip
!unzip -q "/content/Audio_Speech_Actors_01-24.zip" -d "content"
files = glob.glob('ravdess/Actor_*/*.wav')
print(f'Toplam dosya: {len(files)}')

unzip:  cannot find or open /content/Audio_Speech_Actors_01-24.zip, /content/Audio_Speech_Actors_01-24.zip.zip or /content/Audio_Speech_Actors_01-24.zip.ZIP.
Toplam dosya: 0


In [1]:
# 3. Özellik Çıkarımı
import numpy as np
import librosa
def extract_features(path, n_mels=128):
    y, sr = librosa.load(path, sr=16000)
    mels = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels)
    mel_db = librosa.power_to_db(mels)
    delta = librosa.feature.delta(mel_db)
    return np.vstack([mel_db, delta]).T

feat = extract_features(files[0])
print('Özellik şekli:', feat.shape)

NameError: name 'files' is not defined

In [14]:
# 4. Dataset & DataLoader Tanımı
import torch
from torch.utils.data import Dataset, DataLoader
class SERDataset(Dataset):
    def __init__(self, files):
        self.files = files
        self.labels = [int(os.path.basename(f).split('-')[2]) - 1 for f in files]
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        features = extract_features(self.files[idx])
        return torch.tensor(features, dtype=torch.float32), self.labels[idx]
from sklearn.model_selection import train_test_split
train_files, val_files = train_test_split(
    files, test_size=0.2, random_state=42,
    stratify=[os.path.basename(f).split('-')[2] for f in files]
)
train_loader = DataLoader(SERDataset(train_files), batch_size=16, shuffle=True)
val_loader = DataLoader(SERDataset(val_files), batch_size=16)
print('Eğitim örnek:', len(train_files), 'Doğrulama örnek:', len(val_files))

Eğitim örnek: 1152 Doğrulama örnek: 288


In [15]:
# 5. Model (CNN–LSTM) Tanımı
import torch.nn as nn
class CNNLSTM(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.lstm = nn.LSTM(32*32, 128, batch_first=True)
        self.fc = nn.Linear(128, n_classes)
    def forward(self, x):
        x = x.transpose(1,2).unsqueeze(1)
        x = self.cnn(x)
        B,C,Fp,Tp = x.size()
        x = x.view(B,Tp,C*Fp)
        _,(h,_) = self.lstm(x)
        return self.fc(h.squeeze(0))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CNNLSTM(n_classes=8).to(device)
print(model)

CNNLSTM(
  (cnn): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (lstm): LSTM(1024, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=8, bias=True)
)


In [1]:
# 6. Eğitim Döngüsü
import torch.optim as optim
from sklearn.metrics import accuracy_score
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(1,21):
    model.train()
    losses=[]
    preds,labels=[],[]
    for Xb,yb in train_loader:
        Xb,yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(Xb)
        loss = criterion(out,yb)
        loss.backward(); optimizer.step()
        losses.append(loss.item())
        preds.extend(out.argmax(1).cpu().numpy())
        labels.extend(yb.cpu().numpy())
    print(f"Epoch {epoch}: Loss={sum(losses)/len(losses):.4f}, Acc={accuracy_score(labels,preds):.4f}")

NameError: name 'nn' is not defined

In [ ]:
# 7. Değerlendirme
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for Xb,yb in val_loader:
        Xb,yb = Xb.to(device), yb.to(device)
        out = model(Xb)
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(yb.cpu().numpy())
from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(all_labels, all_preds))
print(confusion_matrix(all_labels, all_preds))